# Code Explainer: REST API Quick Start

Quick guide to using Code Explainer via REST API

## 1. Start the Server

In [ ]:
# Start the API server in a terminal:
# uvicorn app:app --reload --host 0.0.0.0 --port 8000

import requests
import time

BASE_URL = "http://localhost:8000/api/v1"

# Test connection
max_retries = 5
for i in range(max_retries):
    try:
        response = requests.get(f"{BASE_URL}/health", timeout=2)
        print("✓ Connected to API")
        print(response.json())
        break
    except requests.exceptions.ConnectionError:
        print(f"Attempt {i+1}/{max_retries}: Waiting for server...")
        time.sleep(2)
    except Exception as e:
        print(f"Error: {e}")
        break

## 2. Single Code Explanation

In [ ]:
import requests
import json

# Code to explain
code = """
def merge_sorted_arrays(arr1, arr2):
    result = []
    i = j = 0
    while i < len(arr1) and j < len(arr2):
        if arr1[i] <= arr2[j]:
            result.append(arr1[i])
            i += 1
        else:
            result.append(arr2[j])
            j += 1
    result.extend(arr1[i:])
    result.extend(arr2[j:])
    return result
"""

# Request explanation
response = requests.post(
    f"{BASE_URL}/explain",
    json={
        "code": code,
        "language": "python",
        "model": "codet5-base",
        "strategy": "vanilla"
    },
    timeout=30
)

if response.status_code == 200:
    result = response.json()
    print("Explanation:")
    print(result['explanation'])
    print(f"\nProcessing time: {result['processing_time']} ms")
    print(f"Model: {result['model_name']}")
else:
    print(f"Error: {response.status_code}")
    print(response.text)

## 3. Batch Processing

In [ ]:
# Multiple code snippets
codes = [
    "x = [1, 2, 3]",
    "result = sum(x) / len(x)",
    "print(f'Mean: {result}')",
    "x_sorted = sorted(x)",
    "median = x_sorted[len(x_sorted)//2]"
]

# Batch request
response = requests.post(
    f"{BASE_URL}/explain/batch",
    json={
        "codes": codes,
        "language": "python",
        "strategy": "vanilla"
    },
    timeout=60
)

if response.status_code == 200:
    result = response.json()
    print(f"Processed: {result['total_processed']} items")
    print(f"Errors: {result.get('total_errors', 0)}")
    print(f"Total time: {result['execution_time_ms']} ms")
    print(f"\nResults:")
    for i, item in enumerate(result['results'], 1):
        print(f"\n{i}. {item['code'][:40]}...")
        print(f"   Status: {item['status']}")
else:
    print(f"Error: {response.status_code}")

## 4. Retrieval-Augmented Explanations

In [ ]:
# Request with retrieval-augmented generation
response = requests.post(
    f"{BASE_URL}/explain",
    json={
        "code": "def binary_search(arr, target):\n    left, right = 0, len(arr) - 1\n    while left <= right:\n        mid = (left + right) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            left = mid + 1\n        else:\n            right = mid - 1\n    return -1",
        "language": "python",
        "strategy": "retrieval-augmented"
    },
    timeout=30
)

if response.status_code == 200:
    result = response.json()
    print(f"Strategy: {result['strategy']}")
    print(f"Explanation: {result['explanation'][:200]}...")
else:
    print(f"Error: {response.status_code}")

## 5. Error Handling

In [ ]:
# Test error cases

# Empty code
print("Test 1: Empty code")
response = requests.post(
    f"{BASE_URL}/explain",
    json={"code": "", "language": "python"}
)
print(f"Status: {response.status_code}")
if response.status_code != 200:
    print(f"Error: {response.json().get('detail', 'Unknown error')}")

print("\nTest 2: Invalid language")
response = requests.post(
    f"{BASE_URL}/explain",
    json={"code": "x = 1", "language": "brainfuck"}
)
print(f"Status: {response.status_code}")
if response.status_code != 200:
    print(f"Error: {response.json().get('detail', 'Unknown error')}")

print("\nTest 3: Batch too large")
response = requests.post(
    f"{BASE_URL}/explain/batch",
    json={"codes": ["x = 1"] * 101}
)
print(f"Status: {response.status_code}")
if response.status_code != 200:
    print(f"Error: {response.json().get('detail', 'Unknown error')}")

## 6. Performance Benchmarking

In [ ]:
import time
import statistics

# Benchmark single request
print("Benchmarking single request...")
times = []

for i in range(5):
    start = time.time()
    response = requests.post(
        f"{BASE_URL}/explain",
        json={
            "code": "x = 42",
            "language": "python"
        },
        timeout=30
    )
    elapsed = time.time() - start
    times.append(elapsed)
    print(f"  Request {i+1}: {elapsed:.3f}s")

print(f"\nStatistics:")
print(f"  Mean: {statistics.mean(times):.3f}s")
print(f"  Median: {statistics.median(times):.3f}s")
print(f"  Std Dev: {statistics.stdev(times):.3f}s")
print(f"  Min: {min(times):.3f}s")
print(f"  Max: {max(times):.3f}s")

## 7. Available Models

In [ ]:
# Get available models
response = requests.get(f"{BASE_URL}/models")

if response.status_code == 200:
    result = response.json()
    print(f"Available Models:")
    print(f"Default: {result.get('default_model')}")
    print(f"\nLoaded:")
    for model in result.get('models', []):
        print(f"  - {model['name']}")
        print(f"    Type: {model.get('type')}")
        print(f"    Languages: {', '.join(model.get('languages', []))}")
        print(f"    Device: {model.get('device')}")
        print(f"    Status: {model.get('status')}")
else:
    print(f"Error: {response.status_code}")